# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shadow-Taslim/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This notebook frames a decision-support project before choosing a model. The lane is provisional through Week 4. Every number below is computed from the anonymized starter data or its reproducible reference pipeline.

## 1. My lane and why

**Lane 2 — Refresh / Content Opportunity Scoring (predefined).** I want to learn which visible pages deserve human review first for refresh, expansion, protection, pruning, or monitoring. This lane fits the available search, content, freshness, and engagement signals; it has a clear weekly decision and action; and the starter slice contains far more plausible candidates than a reviewer can inspect. The starter also shows that a learned ranking can improve on a fixed rule, while leaving the harder future-window and validation work for the warehouse.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Shadow-Taslim/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not Path(REPO_DIR).is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not Path("data/raw").is_dir() and Path.cwd() != Path("/"):
        os.chdir("..")

assert Path("data/raw/content_refresh_anonymized.csv").is_file(), "Starter CSV not found"

LANE = "Refresh / Content Opportunity Scoring"
UNIT_OF_ANALYSIS = "one pseudonymized page at one weekly decision date"
REVIEW_CAPACITY = 50

print("Lane:", LANE)
print("Unit of analysis:", UNIT_OF_ANALYSIS)
print("Weekly human review capacity:", REVIEW_CAPACITY, "pages")

Lane: Refresh / Content Opportunity Scoring
Unit of analysis: one pseudonymized page at one weekly decision date
Weekly human review capacity: 50 pages


## 2. The question: decision, action, cost of a wrong call

**Research question:** Using only signals available at a weekly decision point, which sufficiently visible pages should an SEO content strategist review first for a refresh decision, ranked by observed decline over the following 30 days?

- **Decision:** which 50 pages enter this week's human review queue, rather than treating every declining page as equally urgent.
- **Unit of analysis:** one pseudonymized page at one decision date. The starter gives one trailing-90-day snapshot per page; the warehouse version will use a prior-90-day feature window and a later, non-overlapping 30-day outcome window.
- **Output:** a ranked queue with a priority score, suggested action (refresh, expand, protect, prune, or monitor), reason codes, and a confidence label. It will never edit a page automatically.
- **Who acts:** an SEO content strategist inspects the top candidates, checks context, and assigns an editor where the evidence justifies it.
- **Cost of a wrong call:** a false positive wastes review/editing time and can trigger an unnecessary change; a false negative can leave a visible, weakening page unattended. Because capacity is limited, **Precision@50** is the primary metric, with average precision, recall, and a manual top-20 review as secondary checks.
- **Why data or ML can help:** a transparent rule is the required baseline, but visibility, freshness, position, content, and engagement signals may interact in ways a small hand rule misses. A learned model earns a place only if it beats that baseline on time-aware, client-grouped holdouts while remaining explainable enough for review.

**One-paragraph frame:** For an SEO content strategist deciding which 50 pages to review each week, I will build a ranked action queue from prior-window search, content, and engagement signals, scoring subsequent 30-day decline risk and evaluating Precision@50. A wrong call costs editor capacity or leaves a useful page unreviewed. A plain rule may not capture interacting signals, but it remains the baseline. I will claim only observed, measured, directional, decision-support results.

In [2]:
review_policy = {
    "actor": "SEO content strategist",
    "weekly_capacity_pages": REVIEW_CAPACITY,
    "primary_metric": f"Precision@{REVIEW_CAPACITY}",
    "secondary_checks": ["average precision", "recall", "manual top-20 review"],
    "allowed_actions": ["refresh", "expand", "protect", "prune", "monitor"],
    "automation_boundary": "recommend only; a human approves every action",
}
error_costs = {
    "false_positive": "wasted review/editor time and possible unnecessary edits",
    "false_negative": "a visible weakening page may be left unattended",
}

print(json.dumps({"review_policy": review_policy, "error_costs": error_costs}, indent=2))

{
  "review_policy": {
    "actor": "SEO content strategist",
    "weekly_capacity_pages": 50,
    "primary_metric": "Precision@50",
    "secondary_checks": [
      "average precision",
      "recall",
      "manual top-20 review"
    ],
    "allowed_actions": [
      "refresh",
      "expand",
      "protect",
      "prune",
      "monitor"
    ],
    "automation_boundary": "recommend only; a human approves every action"
  },
  "error_costs": {
    "false_positive": "wasted review/editor time and possible unnecessary edits",
    "false_negative": "a visible weakening page may be left unattended"
  }
}


## 3. Quick look at the data: three supporting results

The code below tests whether this lane has enough candidates, enough exposure to matter, and evidence that ranking method can change review quality.

1. **13,152 pages (43.8% of the starter slice)** are marked `down` and have at least 100 impressions. This candidate pool is much larger than a 50-page weekly review capacity, so prioritization is necessary.
2. Those pages have a **median of 1,620.5 impressions over 90 days**. The group is therefore not limited to zero-demand pages, although impressions alone do not prove business value.
3. On the starter's current-window proxy and client holdout, **Precision@50 is 0.240 for the transparent baseline and 0.740 for the random forest**. That is approximately 12 versus 37 of the top 50 matching the proxy label. It is evidence that the ranking method matters on this slice—not evidence that the same score will transfer to the warehouse or predict a future outcome.

In [3]:
import pandas as pd

starter_path = Path("data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(starter_path)

candidate_mask = (df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)
candidate_count = int(candidate_mask.sum())
candidate_share = float(candidate_mask.mean())
median_impressions = float(df.loc[candidate_mask, "impressions_90d"].median())

print(f"Starter pages: {len(df):,} across {df['client_id'].nunique()} pseudonymized clients")
print(f"1) Declining with >=100 impressions: {candidate_count:,} pages ({candidate_share:.1%})")
print(f"2) Median impressions_90d in that group: {median_impressions:,.1f}")

# The metrics are regenerated only when absent, so a fresh clone also runs cleanly.
results_path = Path("outputs/model_results.json")
if not results_path.is_file():
    subprocess.run([sys.executable, "scripts/run_all.py"], check=True)
results = json.loads(results_path.read_text())
baseline_p50 = results["baseline"]["baseline_precision_at_50"]
forest_p50 = results["models"]["random_forest"]["precision_at_50"]
print(f"3) Client-holdout Precision@50: baseline={baseline_p50:.3f}; random_forest={forest_p50:.3f}")
print(f"   Approximate proxy matches among the top 50: {round(baseline_p50 * 50)} vs {round(forest_p50 * 50)}")

assert candidate_count > REVIEW_CAPACITY, "The ranking problem needs more candidates than weekly capacity"

Starter pages: 30,000 across 32 pseudonymized clients
1) Declining with >=100 impressions: 13,152 pages (43.8%)
2) Median impressions_90d in that group: 1,620.5
3) Client-holdout Precision@50: baseline=0.240; random_forest=0.740
   Approximate proxy matches among the top 50: 12 vs 37


## 4. Careful words: what I can and cannot claim

**What I can claim, if the later validation supports it:** I observed associations between pre-decision signals and later outcomes; I measured whether a ranking beat a transparent baseline on unseen clients and time periods; and I produced directional, decision-support candidates with reasons for human review.

**What I cannot claim:** that a refresh caused recovery, that any feature is a Google ranking factor, that the starter proxy is a future outcome, or that the starter's 0.740 Precision@50 will transfer to the warehouse. I also will not publish client-identifying details, raw URLs, titles, or private queries.

**Validation guardrails:** define magnitude, persistence, and minimum-volume rules before modeling; keep feature and target windows separate; use time-aware and client-grouped holdouts; check seasonality, consolidation, SERP/AI click-loss, and low-volume noise; and review the top recommendations by hand. Under the starter proxy, `trend_direction` and `trend_pct` are target-derived and must not be features. IDs are for grouping and joining only.

In [4]:
starter_proxy_leakage = {"trend_direction", "trend_pct"}
identifier_columns = {"content_id", "client_id"}
example_feature_candidates = {
    "impressions_90d", "clicks_90d", "sessions_90d", "content_age_days",
    "days_since_last_update", "avg_position", "ctr", "engagement_rate",
}
assert not (example_feature_candidates & starter_proxy_leakage)
assert not (example_feature_candidates & identifier_columns)
assert candidate_count > REVIEW_CAPACITY

future_contract = {
    "feature_window": "prior 90 days only",
    "target_window": "following 30 days; no overlap with features",
    "validation": "time-aware and client-grouped holdouts",
    "excluded_from_starter_features": sorted(starter_proxy_leakage | identifier_columns),
    "claim_boundary": "observed, measured, directional, decision-support",
}
print(json.dumps(future_contract, indent=2))
print("PASS: feature exclusions, ranking need, and future-window boundary are explicit.")

{
  "feature_window": "prior 90 days only",
  "target_window": "following 30 days; no overlap with features",
  "validation": "time-aware and client-grouped holdouts",
  "excluded_from_starter_features": [
    "client_id",
    "content_id",
    "trend_direction",
    "trend_pct"
  ],
  "claim_boundary": "observed, measured, directional, decision-support"
}
PASS: feature exclusions, ranking need, and future-window boundary are explicit.


## Self-check

- [x] Every section is filled with markdown framing and supporting code.
- [x] The notebook runs top to bottom without errors.
- [x] No client names, data URLs, titles, or private queries are included.
- [x] Claims use careful words: observed, measured, directional, and decision-support.
- [x] The decision, human action, unit of analysis, output, error costs, and success metric are explicit.
- [x] Committed under `work/notebooks/w01_research_question.ipynb`; submit the repository URL on the assignment card.